# **Часть 1. Знакомство с данными в PostgreSQL**


#### **Задание 1.1: загрузите и изучите данные**


In [1]:
pip install psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# --- Подключение ---
host = "rc1d-pdq3kahro9enkfir.mdb.yandexcloud.net"
port = 6432
database = "playground_ds_20260915_7e2a105f57"
user = "ds_20260915_7e2a105f57"
password = "f93cb2a8d1c74fab8ecc9e9b3c701ffe"

conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(conn_str)

with engine.connect() as conn:
    print("✅ Соединение:", conn.execute(text("SELECT 1")).fetchone())


✅ Соединение: (1,)


In [3]:
df_customers = pd.read_sql_query("SELECT * FROM customers", engine)
df_sessions  = pd.read_sql_query("SELECT * FROM sessions",  engine)
df_events    = pd.read_sql_query("SELECT * FROM events",    engine)
df_orders    = pd.read_sql_query("SELECT * FROM orders",    engine)

print(f"customers: {df_customers.shape}")
print(f"sessions:  {df_sessions.shape}")
print(f"events:    {df_events.shape}")
print(f"orders:    {df_orders.shape}")


customers: (17846, 7)
sessions:  (37699, 6)
events:    (239089, 10)
orders:    (10652, 10)


In [4]:
# Уникальные идентификаторы
print("=== Уникальные идентификаторы ===")
print(f"customer_id в customers: {df_customers['customer_id'].nunique()}")
print(f"customer_id в sessions:  {df_sessions['customer_id'].nunique()}")
print(f"customer_id в orders:     {df_orders['customer_id'].nunique()}")
print(f"session_id в sessions:   {df_sessions['session_id'].nunique()}")
print(f"session_id в events:      {df_events['session_id'].nunique()}")
print(f"event_id в events:        {df_events['event_id'].nunique()}")
print(f"order_id в orders:        {df_orders['order_id'].nunique()}")

# Дубликаты по ключам
print("\n=== Дубликаты по ключам ===")
for name, df, key in [
    ("customers", df_customers, "customer_id"),
    ("sessions",  df_sessions,  "session_id"),
    ("events",    df_events,    "event_id"),
    ("orders",    df_orders,    "order_id"),
]:
    dup_count = len(df) - df[key].nunique()
    print(f"{name}: строк {len(df)}, уникальных {key} {df[key].nunique()}, дубликатов {dup_count}")


=== Уникальные идентификаторы ===
customer_id в customers: 17846
customer_id в sessions:  16894
customer_id в orders:     8230
session_id в sessions:   37699
session_id в events:      37699
event_id в events:        239089
order_id в orders:        10652

=== Дубликаты по ключам ===
customers: строк 17846, уникальных customer_id 17846, дубликатов 0
sessions: строк 37699, уникальных session_id 37699, дубликатов 0
events: строк 239089, уникальных event_id 239089, дубликатов 0
orders: строк 10652, уникальных order_id 10652, дубликатов 0


In [5]:
# Приводим временные поля к datetime
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'], errors='coerce')
df_sessions['start_time']   = pd.to_datetime(df_sessions['start_time'],   errors='coerce')
df_events['timestamp']      = pd.to_datetime(df_events['timestamp'],      errors='coerce')
df_orders['order_time']     = pd.to_datetime(df_orders['order_time'],     errors='coerce')

print("=== Минимальные и максимальные даты ===")
print(f"signup_date:  min={df_customers['signup_date'].min()}  max={df_customers['signup_date'].max()}")
print(f"start_time:   min={df_sessions['start_time'].min()}   max={df_sessions['start_time'].max()}")
print(f"timestamp:    min={df_events['timestamp'].min()}       max={df_events['timestamp'].max()}")
print(f"order_time:   min={df_orders['order_time'].min()}     max={df_orders['order_time'].max()}")

# Пропуски во временных полях
print("\n=== Пропуски во временных полях ===")
print(f"signup_date: {df_customers['signup_date'].isna().sum()}")
print(f"start_time:  {df_sessions['start_time'].isna().sum()}")
print(f"timestamp:   {df_events['timestamp'].isna().sum()}")
print(f"order_time:  {df_orders['order_time'].isna().sum()}")


=== Минимальные и максимальные даты ===
signup_date:  min=2020-01-01 00:00:00  max=2025-10-31 00:00:00
start_time:   min=2024-01-01 00:57:40   max=2025-10-31 23:34:11
timestamp:    min=2024-01-01 01:02:40       max=2025-11-01 01:50:04
order_time:   min=2024-01-01 07:08:06     max=2025-10-31 22:59:41

=== Пропуски во временных полях ===
signup_date: 0
start_time:  0
timestamp:   0
order_time:  0


In [6]:
# Пропуски в ключевых полях
print("=== Пропуски в ключевых полях ===")
print("customers:", df_customers[['customer_id', 'signup_date']].isna().sum().to_dict())
print("sessions: ", df_sessions[['session_id', 'customer_id', 'start_time']].isna().sum().to_dict())
print("events:   ", df_events[['event_id', 'session_id', 'timestamp', 'event_type', 'product_id']].isna().sum().to_dict())
print("orders:   ", df_orders[['order_id', 'customer_id', 'order_time', 'total_usd']].isna().sum().to_dict())

# Связи между таблицами
print("\n=== Связи между таблицами ===")
sessions_cust = set(df_sessions['customer_id'].dropna().unique())
customers_cust = set(df_customers['customer_id'].unique())
print(f"customer_id из sessions без соответствия в customers: {len(sessions_cust - customers_cust)}")

events_sess = set(df_events['session_id'].dropna().unique())
sessions_sess = set(df_sessions['session_id'].unique())
print(f"session_id из events без соответствия в sessions: {len(events_sess - sessions_sess)}")

orders_cust = set(df_orders['customer_id'].dropna().unique())
print(f"customer_id из orders без соответствия в customers: {len(orders_cust - customers_cust)}")

# Пользователи во всех таблицах
cust_all = customers_cust & sessions_cust & orders_cust
print(f"Пользователей во всех таблицах: {len(cust_all)}")


=== Пропуски в ключевых полях ===
customers: {'customer_id': 0, 'signup_date': 0}
sessions:  {'session_id': 0, 'customer_id': 0, 'start_time': 0}
events:    {'event_id': 0, 'session_id': 0, 'timestamp': 0, 'event_type': 0, 'product_id': 24881}
orders:    {'order_id': 0, 'customer_id': 0, 'order_time': 0, 'total_usd': 0}

=== Связи между таблицами ===
customer_id из sessions без соответствия в customers: 0
session_id из events без соответствия в sessions: 0
customer_id из orders без соответствия в customers: 0
Пользователей во всех таблицах: 8230


In [7]:
print("=== Воронка событий ===")
funnel = ['page_view', 'add_to_cart', 'checkout', 'purchase']
counts = df_events['event_type'].value_counts()
for et in funnel:
    c = counts.get(et, 0)
    print(f"  {et:15s}: {c:>8d}  ({c/len(df_events)*100:.1f}%)")

pv  = counts.get('page_view', 0)
atc = counts.get('add_to_cart', 0)
co  = counts.get('checkout', 0)
pu  = counts.get('purchase', 0)

print("\n=== Конверсии ===")
print(f"page_view -> add_to_cart:   {atc/pv:.4f} ({atc/pv*100:.2f}%)"  if pv else "page_view -> add_to_cart: 0")
print(f"add_to_cart -> checkout:    {co/atc:.4f} ({co/atc*100:.2f}%)" if atc else "add_to_cart -> checkout: 0")
print(f"checkout -> purchase:       {pu/co:.4f} ({pu/co*100:.2f}%)"  if co else "checkout -> purchase: 0")


=== Воронка событий ===
  page_view      :   169300  (70.8%)
  add_to_cart    :    44908  (18.8%)
  checkout       :    14229  (6.0%)
  purchase       :    10652  (4.5%)

=== Конверсии ===
page_view -> add_to_cart:   0.2653 (26.53%)
add_to_cart -> checkout:    0.3168 (31.68%)
checkout -> purchase:       0.7486 (74.86%)


In [8]:
print("""
=== Ключевые наблюдения по данным ===

1. events не содержит customer_id — нужно подтягивать через session_id из sessions.
2. Денежные признаки берутся из orders (total_usd), а не из events.
3. Для конверсии add_to_cart -> purchase используем события purchase из events.
4. product_id нужен только для признака уникальных товаров; строки без product_id исключаются.
5. Длина сессии = (max timestamp - min timestamp) по events в рамках session_id.
6. Все временные поля приведены к datetime UTC.
7. Дубликаты по event_id нужно удалить до расчёта.
""")



=== Ключевые наблюдения по данным ===

1. events не содержит customer_id — нужно подтягивать через session_id из sessions.
2. Денежные признаки берутся из orders (total_usd), а не из events.
3. Для конверсии add_to_cart -> purchase используем события purchase из events.
4. product_id нужен только для признака уникальных товаров; строки без product_id исключаются.
5. Длина сессии = (max timestamp - min timestamp) по events в рамках session_id.
6. Все временные поля приведены к datetime UTC.
7. Дубликаты по event_id нужно удалить до расчёта.



In [9]:
run_date = pd.Timestamp("2025-09-01")
print(f"run_date = {run_date.date()}")

# --- 1. Копии ---
cust = df_customers.copy()
sess = df_sessions.copy()
evt  = df_events.copy()
ord  = df_orders.copy()

# --- 2. Приведение временных полей ---
cust['signup_date'] = pd.to_datetime(cust['signup_date'], errors='coerce')
sess['start_time']  = pd.to_datetime(sess['start_time'],  errors='coerce')
evt['timestamp']    = pd.to_datetime(evt['timestamp'],    errors='coerce')
ord['order_time']   = pd.to_datetime(ord['order_time'],   errors='coerce')

# --- 3. Удаление строк с невалидными временными метками ---
cust = cust[cust['signup_date'].notna()]
sess = sess[sess['start_time'].notna()]
evt  = evt[evt['timestamp'].notna()]
ord  = ord[ord['order_time'].notna()]

# --- 4. Удаление дубликатов по ключам ---
sess = sess.drop_duplicates(subset=['session_id'])
evt  = evt.drop_duplicates(subset=['event_id'])
ord  = ord.drop_duplicates(subset=['order_id'])

# --- 5. Фильтрация по run_date (только данные ДО run_date) ---
cust = cust[cust['signup_date'] < run_date]
sess = sess[sess['start_time'] < run_date]
evt  = evt[evt['timestamp'] < run_date]
ord  = ord[ord['order_time'] < run_date]

# --- ВАЖНО: сначала merge с sessions, чтобы у всех событий был customer_id ---
evt = evt.merge(
    sess[['session_id', 'customer_id']],
    on='session_id', how='left'
)

# Теперь можно безопасно фильтровать по product_id — customer_id уже есть
evt_with_product = evt[evt['product_id'].notna()].copy()

# --- 6. total_usd: приводим к числу, NaN и ошибки -> 0.0 ---
ord['total_usd'] = pd.to_numeric(ord['total_usd'], errors='coerce').fillna(0.0)

print(f"После предобработки:")
print(f"  customers: {len(cust)}")
print(f"  sessions:  {len(sess)}")
print(f"  events:    {len(evt)}")
print(f"  orders:    {len(ord)}")
print(f"  events с product_id: {len(evt_with_product)}")


run_date = 2025-09-01
После предобработки:
  customers: 17292
  sessions:  34206
  events:    216603
  orders:    9645
  events с product_id: 194062


In [10]:
# Длина сессии = разница между последним и первым событием в сессии (в секундах)
session_len = (
    evt.groupby('session_id')['timestamp']
    .agg(['min', 'max'])
    .assign(session_length_sec=lambda x: (x['max'] - x['min']).dt.total_seconds())
    [['session_length_sec']]
    .reset_index()
)

# Объединяем с sessions для получения customer_id и start_time
sess_full = sess.merge(session_len, on='session_id', how='left')
sess_full['session_length_sec'] = sess_full['session_length_sec'].fillna(0.0)

print(f"Сессий с рассчитанной длиной: {sess_full['session_length_sec'].notna().sum()}")
print(f"Сессий с длиной 0 (нет событий): {(sess_full['session_length_sec'] == 0).sum()}")
print(f"Средняя длина сессии (сек): {sess_full['session_length_sec'].mean():.1f}")


Сессий с рассчитанной длиной: 34206
Сессий с длиной 0 (нет событий): 3130
Средняя длина сессии (сек): 3571.7


In [11]:
def filter_window(df, ts_col, run_date, days):
    """Возвращает строки в окне [run_date - days, run_date)."""
    start = run_date - pd.Timedelta(days=days)
    return df[(df[ts_col] >= start) & (df[ts_col] < run_date)]

# --- Окна по всем событиям (с customer_id) ---
evt_7  = filter_window(evt,  'timestamp', run_date, 7)
evt_30 = filter_window(evt,  'timestamp', run_date, 30)

# --- Окна только по событиям с product_id (для уникальных товаров) ---
# customer_id здесь уже есть благодаря merge выше
evt_prod_7  = filter_window(evt_with_product, 'timestamp', run_date, 7)
evt_prod_30 = filter_window(evt_with_product, 'timestamp', run_date, 30)

# --- 1-2. Количество page_view и add_to_cart ---
def count_event(df, event_type):
    return (
        df[df['event_type'] == event_type]
        .groupby('customer_id')
        .size()
    )

page_view_7  = count_event(evt_7,  'page_view')
page_view_30 = count_event(evt_30, 'page_view')
add_to_cart_7  = count_event(evt_7,  'add_to_cart')
add_to_cart_30 = count_event(evt_30, 'add_to_cart')
purchase_7  = count_event(evt_7,  'purchase')
purchase_30 = count_event(evt_30, 'purchase')

# --- 3. Конверсия page_view -> add_to_cart ---
conv_pv_atc_7  = (add_to_cart_7  / page_view_7).replace([np.inf, -np.inf], 0.0).fillna(0.0)
conv_pv_atc_30 = (add_to_cart_30 / page_view_30).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# --- 4. Конверсия add_to_cart -> purchase ---
conv_atc_pu_7  = (purchase_7  / add_to_cart_7).replace([np.inf, -np.inf], 0.0).fillna(0.0)
conv_atc_pu_30 = (purchase_30 / add_to_cart_30).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# --- 5. Уникальные product_id ---
unique_products_7 = (
    evt_prod_7.groupby('customer_id')['product_id'].nunique()
)
unique_products_30 = (
    evt_prod_30.groupby('customer_id')['product_id'].nunique()
)

print("Признаки из events рассчитаны")
print(f"  Пользователей с page_view (7д): {page_view_7.shape[0]}")
print(f"  Пользователей с add_to_cart (7д): {add_to_cart_7.shape[0]}")
print(f"  Пользователей с purchase (7д): {purchase_7.shape[0]}")

Признаки из events рассчитаны
  Пользователей с page_view (7д): 374
  Пользователей с add_to_cart (7д): 265
  Пользователей с purchase (7д): 117


In [12]:
# --- 6. Средняя длина сессии за 30 дней ---
sess_30 = filter_window(sess_full, 'start_time', run_date, 30)
avg_session_len_30 = (
    sess_30.groupby('customer_id')['session_length_sec'].mean()
)

# --- 7. Количество сессий за 7 и 30 дней ---
sess_7  = filter_window(sess_full, 'start_time', run_date, 7)
sessions_count_7  = sess_7.groupby('customer_id').size()
sessions_count_30 = sess_30.groupby('customer_id').size()

print("Признаки из sessions рассчитаны")
print(f"  Пользователей с сессиями (7д): {sessions_count_7.shape[0]}")
print(f"  Пользователей с сессиями (30д): {sessions_count_30.shape[0]}")


Признаки из sessions рассчитаны
  Пользователей с сессиями (7д): 370
  Пользователей с сессиями (30д): 1677


In [13]:
ord_30 = filter_window(ord, 'order_time', run_date, 30)

# --- 8. Число дней с последней покупки до run_date ---
# Сначала считаем последнюю покупку по ВСЕМ заказам до run_date (не только 30д)
last_purchase_all = ord.groupby('customer_id')['order_time'].max()
days_since_last_purchase = (run_date - last_purchase_all).dt.days

# Для пользователей без заказов вообще ставим -1
all_cust_ids = cust['customer_id'].values
days_since_last_purchase = days_since_last_purchase.reindex(all_cust_ids, fill_value=-1)

# --- 9. Количество заказов за 30 дней ---
order_count_30 = ord_30.groupby('customer_id').size()

# --- 10. Сумма total_usd за 30 дней ---
total_usd_sum_30 = ord_30.groupby('customer_id')['total_usd'].sum()

# --- 11. Средний total_usd за 30 дней ---
total_usd_avg_30 = ord_30.groupby('customer_id')['total_usd'].mean()

print("Признаки из orders рассчитаны")
print(f"  Пользователей с заказами (30д): {order_count_30.shape[0]}")
print(f"  Пользователей без заказов вообще: {(days_since_last_purchase == -1).sum()}")

Признаки из orders рассчитаны
  Пользователей с заказами (30д): 503
  Пользователей без заказов вообще: 9888


In [14]:
# Базовый набор customer_id — все пользователи, зарегистрированные до run_date
features = pd.DataFrame({'customer_id': all_cust_ids})
features['run_date'] = run_date.date()

# Заполняем признаки, выравнивая по customer_id
def align(series, fill_value):
    return series.reindex(all_cust_ids, fill_value=fill_value).values

# Счётчики (int, fill 0)
features['page_view_count_7d']        = align(page_view_7, 0)
features['page_view_count_30d']       = align(page_view_30, 0)
features['add_to_cart_count_7d']      = align(add_to_cart_7, 0)
features['add_to_cart_count_30d']     = align(add_to_cart_30, 0)
features['unique_product_id_count_7d']  = align(unique_products_7, 0)
features['unique_product_id_count_30d'] = align(unique_products_30, 0)
features['sessions_count_7d']        = align(sessions_count_7, 0)
features['sessions_count_30d']       = align(sessions_count_30, 0)
features['order_count_30d']           = align(order_count_30, 0)

# Вещественные (float, fill 0.0)
features['conversion_pv_to_atc_7d']   = align(conv_pv_atc_7, 0.0)
features['conversion_pv_to_atc_30d']  = align(conv_pv_atc_30, 0.0)
features['conversion_atc_to_purchase_7d']  = align(conv_atc_pu_7, 0.0)
features['conversion_atc_to_purchase_30d'] = align(conv_atc_pu_30, 0.0)
features['avg_session_length_30d']   = align(avg_session_len_30, 0.0)
features['total_usd_sum_30d']         = align(total_usd_sum_30, 0.0)
features['total_usd_avg_30d']         = align(total_usd_avg_30, 0.0)

# Особый случай: days_since_last_purchase (fill -1)
features['days_since_last_purchase'] = days_since_last_purchase.values

# Порядок колонок
COLUMN_ORDER = [
    'customer_id', 'run_date',
    'page_view_count_7d',
    'page_view_count_30d',
    'add_to_cart_count_7d',
    'add_to_cart_count_30d',
    'conversion_pv_to_atc_7d',
    'conversion_pv_to_atc_30d',
    'conversion_atc_to_purchase_7d',
    'conversion_atc_to_purchase_30d',
    'unique_product_id_count_7d',
    'unique_product_id_count_30d',
    'avg_session_length_30d',
    'sessions_count_7d',
    'sessions_count_30d',
    'days_since_last_purchase',
    'order_count_30d',
    'total_usd_sum_30d',
    'total_usd_avg_30d',
]
features = features[COLUMN_ORDER]

print(f"Итоговая таблица: {features.shape}")
features.head(10)


Итоговая таблица: (17292, 19)


,customer_id,run_date,page_view_count_7d,page_view_count_30d,add_to_cart_count_7d,add_to_cart_count_30d,conversion_pv_to_atc_7d,conversion_pv_to_atc_30d,conversion_atc_to_purchase_7d,conversion_atc_to_purchase_30d,unique_product_id_count_7d,unique_product_id_count_30d,avg_session_length_30d,sessions_count_7d,sessions_count_30d,days_since_last_purchase,order_count_30d,total_usd_sum_30d,total_usd_avg_30d
0,1,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,67,0,0.0,0.0
1,2,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,241,0,0.0,0.0
2,3,2025-09-01,8,8,0,0,0.0,0.0,0.0,0.0,8,8,7140.0,1,1,-1,0,0.0,0.0
3,4,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
4,5,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,213,0,0.0,0.0
5,6,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,277,0,0.0,0.0
6,7,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
7,9,2025-09-01,0,4,0,0,0.0,0.0,0.0,0.0,0,4,2280.0,0,1,467,0,0.0,0.0
8,10,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
9,12,2025-09-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,392,0,0.0,0.0


In [15]:
print("=== Проверка: нет данных из будущего ===")

# Проверяем, что все отфильтрованные данные строго меньше run_date
assert (sess['start_time'] < run_date).all(), "В sessions есть данные >= run_date!"
assert (evt['timestamp'] < run_date).all(), "В events есть данные >= run_date!"
assert (ord['order_time'] < run_date).all(), "В orders есть данные >= run_date!"
assert (cust['signup_date'] < run_date).all(), "В customers есть signup_date >= run_date!"
print("✅ Все временные фильтры соблюдены — данные из будущего исключены")

# Проверка окон
max_event_ts_7 = evt_7['timestamp'].max() if len(evt_7) > 0 else None
max_event_ts_30 = evt_30['timestamp'].max() if len(evt_30) > 0 else None
print(f"  Макс timestamp в окне 7д:  {max_event_ts_7}")
print(f"  Макс timestamp в окне 30д: {max_event_ts_30}")
print(f"  Окно 7д должно быть в [{(run_date - pd.Timedelta(days=7)).date()}, {run_date.date()})")
print(f"  Окно 30д должно быть в [{(run_date - pd.Timedelta(days=30)).date()}, {run_date.date()})")


=== Проверка: нет данных из будущего ===
✅ Все временные фильтры соблюдены — данные из будущего исключены
  Макс timestamp в окне 7д:  2025-08-31 23:56:13
  Макс timestamp в окне 30д: 2025-08-31 23:56:13
  Окно 7д должно быть в [2025-08-25, 2025-09-01)
  Окно 30д должно быть в [2025-08-02, 2025-09-01)


In [16]:
print("=== Проверка: уникальность customer_id ===")
dups = features.duplicated(subset=['customer_id', 'run_date']).sum()
print(f"Дубликатов (customer_id, run_date): {dups}")
assert dups == 0, "Есть дубликаты!"
print(f"✅ Одна строка на пару customer_id + run_date: {len(features)} строк")


=== Проверка: уникальность customer_id ===
Дубликатов (customer_id, run_date): 0
✅ Одна строка на пару customer_id + run_date: 17292 строк


In [17]:
print("=== Проверка: все признаки присутствуют ===")
required = set(COLUMN_ORDER) - {'customer_id', 'run_date'}
present = set(features.columns) - {'customer_id', 'run_date'}
missing = required - present
extra = present - required
print(f"Требуется признаков: {len(required)}")
print(f"Присутствует: {len(present)}")
print(f"Отсутствует: {missing if missing else 'нет'}")
print(f"Лишних: {extra if extra else 'нет'}")
assert not missing, f"Не хватает признаков: {missing}"
print("✅ Все 17 признаков присутствуют")


=== Проверка: все признаки присутствуют ===
Требуется признаков: 17
Присутствует: 17
Отсутствует: нет
Лишних: нет
✅ Все 17 признаков присутствуют


In [18]:
print("=== Проверка: деление на ноль и пропуски ===")

# Конверсии: где page_view=0, конверсия должна быть 0.0
pv_zero = features[features['page_view_count_7d'] == 0]
assert (pv_zero['conversion_pv_to_atc_7d'] == 0.0).all(), \
    "Конверсия pv->atc не 0 при page_view=0!"
print(f"✅ Конверсия pv->atc=0 при page_view=0 (проверено на {len(pv_zero)} строках)")

atc_zero = features[features['add_to_cart_count_7d'] == 0]
assert (atc_zero['conversion_atc_to_purchase_7d'] == 0.0).all(), \
    "Конверсия atc->purchase не 0 при add_to_cart=0!"
print(f"✅ Конверсия atc->purchase=0 при add_to_cart=0 (проверено на {len(atc_zero)} строках)")

# days_since_last_purchase: -1 для пользователей БЕЗ ЗАКАЗОВ ВООБЩЕ
# (а не просто без заказов за 30 дней)
no_orders_all = features[features['days_since_last_purchase'] == -1]
# У них действительно не должно быть заказов ни в каком окне
assert (no_orders_all['order_count_30d'] == 0).all(), \
    "У пользователей с days_since_last_purchase=-1 есть заказы за 30 дней!"
print(f"✅ days_since_last_purchase=-1 корректно для пользователей без заказов ({len(no_orders_all)} строк)")

# Нет NaN в итоговой таблице
na_count = features.isna().sum().sum()
print(f"NaN в итоговой таблице: {na_count}")
assert na_count == 0, f"Есть NaN: {features.isna().sum()[features.isna().sum() > 0]}"
print("✅ Пропусков в итоговой таблице нет")

=== Проверка: деление на ноль и пропуски ===
✅ Конверсия pv->atc=0 при page_view=0 (проверено на 16924 строках)
✅ Конверсия atc->purchase=0 при add_to_cart=0 (проверено на 17030 строках)
✅ days_since_last_purchase=-1 корректно для пользователей без заказов (9888 строк)
NaN в итоговой таблице: 0
✅ Пропусков в итоговой таблице нет


In [19]:
print("=== Проверка: разумность значений ===")
print("\nСтатистика по счётчикам:")
count_cols = [
    'page_view_count_7d', 'page_view_count_30d',
    'add_to_cart_count_7d', 'add_to_cart_count_30d',
    'sessions_count_7d', 'sessions_count_30d',
    'order_count_30d', 'unique_product_id_count_7d', 'unique_product_id_count_30d',
]
print(features[count_cols].describe().round(2))

print("\nСтатистика по конверсиям:")
conv_cols = [
    'conversion_pv_to_atc_7d', 'conversion_pv_to_atc_30d',
    'conversion_atc_to_purchase_7d', 'conversion_atc_to_purchase_30d',
]
print(features[conv_cols].describe().round(4))

print("\nСтатистика по денежным признакам:")
money_cols = ['total_usd_sum_30d', 'total_usd_avg_30d', 'days_since_last_purchase']
print(features[money_cols].describe().round(2))

print("\nСтатистика по длине сессий:")
print(features['avg_session_length_30d'].describe().round(1))

# Конверсии должны быть в [0, 1] (или близко — purchase может быть > add_to_cart теоретически)
for col in conv_cols:
    mn, mx = features[col].min(), features[col].max()
    print(f"  {col}: min={mn:.4f}, max={mx:.4f}")

# Дни с последней покупки: все >= -1
assert features['days_since_last_purchase'].min() >= -1
print("\n✅ Все значения выглядят разумно")


=== Проверка: разумность значений ===

Статистика по счётчикам:
       page_view_count_7d  page_view_count_30d  add_to_cart_count_7d  \
count            17292.00             17292.00              17292.00   
mean                 0.09                 0.44                  0.03   
std                  0.72                 1.56                  0.24   
min                  0.00                 0.00                  0.00   
25%                  0.00                 0.00                  0.00   
50%                  0.00                 0.00                  0.00   
75%                  0.00                 0.00                  0.00   
max                 11.00                15.00                  5.00   

       add_to_cart_count_30d  sessions_count_7d  sessions_count_30d  \
count               17292.00           17292.00            17292.00   
mean                    0.12               0.02                0.10   
std                     0.52               0.14                0.31   
min

In [20]:
# Сохраняем для сравнения с .py-модулем
features.to_parquet("features_2025-09-01.parquet", index=False)
print(f"Сохранено: features_2025-09-01.parquet ({features.shape})")

# Финальная сводка
print(f"\n=== ИТОГ ===")
print(f"run_date: {run_date.date()}")
print(f"Строк: {len(features)}")
print(f"Признаков: {len(COLUMN_ORDER) - 2}")
print(f"Колонки: {COLUMN_ORDER}")


Сохранено: features_2025-09-01.parquet ((17292, 19))

=== ИТОГ ===
run_date: 2025-09-01
Строк: 17292
Признаков: 17
Колонки: ['customer_id', 'run_date', 'page_view_count_7d', 'page_view_count_30d', 'add_to_cart_count_7d', 'add_to_cart_count_30d', 'conversion_pv_to_atc_7d', 'conversion_pv_to_atc_30d', 'conversion_atc_to_purchase_7d', 'conversion_atc_to_purchase_30d', 'unique_product_id_count_7d', 'unique_product_id_count_30d', 'avg_session_length_30d', 'sessions_count_7d', 'sessions_count_30d', 'days_since_last_purchase', 'order_count_30d', 'total_usd_sum_30d', 'total_usd_avg_30d']
